In [7]:
import os
import json
import pandas as pd
from scipy.stats import pearsonr

data_dir = './CWB_OCB_Merged_Results' 
all_data = []

for filename in os.listdir(data_dir):
    if filename.endswith('.json') and filename.startswith('psi_'):
        file_path = os.path.join(data_dir, filename)

        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        if isinstance(data, dict):
            data = [data]

        for entry in data:
            all_data.append({
                "file": filename,
                "Extraversion": entry.get("Extraversion"),
                "Agreeableness": entry.get("Agreeableness"),
                "Conscientiousness": entry.get("Conscientiousness"),
                "Neuroticism": entry.get("Neuroticism"),
                "Openness": entry.get("Openness"),
                "OCB": entry.get("OCB"),
                "CWB": entry.get("CWB")
            })

df_all = pd.DataFrame(all_data)

traits = ["Extraversion", "Agreeableness", "Conscientiousness", "Neuroticism", "Openness"]
targets = ["OCB", "CWB"]

results = []

for filename in df_all["file"].unique():
    df_subset = df_all[df_all["file"] == filename]
    
    for target in targets:
        row = {"file": filename, "target": target}
        for trait in traits:
            df_valid = df_subset[[trait, target]].dropna()
            if len(df_valid) > 2:
                corr, _ = pearsonr(df_valid[trait], df_valid[target])
                row[trait] = corr
            else:
                row[trait] = None
        results.append(row)

df_results = pd.DataFrame(results)
df_results.to_csv("./Output/correlation_by_file.csv", index=False, encoding='utf-8-sig')

print("./Output/correlation_by_file.csv")


./Output/correlation_by_file.csv
